In [ ]:
import tensorflow as tf

In [ ]:
# function made in 02_01 notebook

import matplotlib.pyplot as plt
import numpy as np

def plot_decision_boundary(model,x,y):
  # plots the decision boundary created by a model predicting on x.

  # define axis boundaries of plot and create meshgrid
  x_min, x_max = x[:, 0].min()-0.1, x[:,0].max()+0.1 # 0.1 is to have breathing room for the min and max points
  y_min, y_max = x[:, 1].min()-0.1, x[:,1].max()+0.1 # Corrected y_min, y_max to use x[:,1]

  # Create a meshgrid from both linspace arrays
  xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))

  # create x values to make predictions on
  x_in = np.c_[xx.ravel(), yy.ravel()] # np.c_ is a shorthand for the actual funct

  # Make predictions (these will be logits since the last layer is Dense(1) without activation)
  y_pred = model.predict(x_in)

  # check for multi-class (multiple labes for a single input)
  if len(y_pred[0]) > 1:
    print("doing multiclass classification")
    # we have to reshape our predictions to get it ready for plotting
    y_pred = np.argmax(y_pred,axis=1).reshape(xx.shape)
  else:
    print("doing binary classification")
    y_pred = np.round(y_pred).reshape(xx.shape)

  plot_data = y_pred

  # plot the decision boundary
  plt.contourf(xx,yy, plot_data.reshape(xx.shape), cmap = plt.cm.RdYlBu, alpha = 0.7)
  plt.scatter(x[:,0], x[:,1], c = y, s = 40, cmap = plt.cm.RdYlBu)
  plt.xlim(xx.min(), xx.max())
  plt.ylim(yy.min(), yy.max())
  plt.show() # Ensure plot is displayed

In [ ]:
from sklearn.datasets import make_circles

# make 1000 examples
n_samples = 1000

# create circles
x, y = make_circles(n_samples,
                    noise = 0.03)

In [ ]:
x,y

In [ ]:
x[0].shape # this our input shape i.e. what we feed into the model

Introducing Non-Linearity to our model

In [ ]:
model_5 = tf.keras.Sequential([
    tf.keras.layers.Dense(1, activation="relu")
])

model_5.compile(loss = "BinaryCrossentropy",
                optimizer = tf.keras.optimizers.Adam(learning_rate = 0.001),
                metrics = ["accuracy"])

history = model_5.fit(x,y,epochs = 100, verbose=0)

In [ ]:
model_5.evaluate(x,y)

# Our model is still doing worse than guessing
# aboslutely no improvement from model_3 even though we change to a non-linear activation function

In [ ]:
# what if we change the nuumber or neurons, layers in conjunction with our activation funct?

model_6 = tf.keras.Sequential([
    tf.keras.layers.Dense(4, activation="relu", input_shape=(2,)), # Explicitly define input shape
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid") # sigmoid gives a value of either 0 or 1, relu uses larger numbers to learn the pattern but cant give a deterministic value
])
model_6.compile( loss = "BinaryCrossentropy",
                optimizer = tf.keras.optimizers.Adam(learning_rate = 0.01),
                metrics = ["accuracy"])

model_6.fit(x,y,epochs = 100, verbose=0)

In [ ]:
model_6.evaluate(x,y)
# this model works!

In [ ]:
# visualise our predictions using the function we made earlier
plot_decision_boundary(model_6,x,y)

In [ ]:
A = tf.cast(tf.range(-10,10), tf.float32)

In [ ]:
plt.plot(A)

In [ ]:
# lets replicate sigmoid: sigmoid(x) = 1/(1+exp(-x)))

def sigmoid(x):
  return 1/(1+tf.exp(-x))

# use sigmoid on our toy tensor

A_sigmoid = sigmoid(A)
A_sigmoid

In [ ]:
plt.plot(A_sigmoid)

In [ ]:
# lets recreate a relu function
def relu(x):
  return tf.maximum(0,x)

relu_A = relu(A)
relu_A

In [ ]:
plt.plot(relu_A)

In [ ]:
# linear activatio function returns the tensor unmodified

A == tf.keras.activations.linear(A)
plt.plot(tf.keras.activations.linear(A))

### This is why our earlier models coulding understand the circular patterns using the linear activation

Eval and Improve our classification model

In [ ]:
# so far we have been training and perdicting on the same dataset, which is not ideal.

len(x), len(y)


In [ ]:
# split x,y to training and testing
x_train, y_train = x[:800], y[:800]
x_test, y_test = x[800:], y[800:]

x_train.shape, x_test.shape, y_train.shape, y_test.shape


In [ ]:
# create a model to fit on the training data and evaluate on the testing data

model_7 = tf.keras.Sequential([
    tf.keras.layers.Dense(4, activation="relu", input_shape=[2,]),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model_7.compile(loss = "BinaryCrossentropy",
                optimizer = tf.keras.optimizers.Adam(learning_rate = 0.01),
                metrics = ["accuracy"]
)

history_model_7 = model_7.fit(x_train, y_train, epochs = 25, verbose=0)


In [ ]:
model_7.summary()

In [ ]:
model_7.evaluate(x_test, y_test) # bang on!

In [ ]:
plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.title("Training")
plot_decision_boundary(model_7, x_train, y_train)
plt.subplot(1,2,2)
plt.title("Testing")
plot_decision_boundary(model_7,  x_test, y_test)
plt.show()

Understanding a model's loss curves

In [ ]:
# a history object returnsa model's training loss values

history_model_7.history # this is for model_7
# it gives us the loss at each epoch

In [ ]:
import pandas as pd
# convert history object into a dataframe

pd.DataFrame(history_model_7.history)

In [ ]:
# plot the plot curve
pd.DataFrame(history_model_7.history).plot()
plt.title("Model 7 loss curves");

Utilising callback to optimize learning rate

In [ ]:
# create a new model

model_8 = tf.keras.Sequential([
    tf.keras.layers.Dense(4, activation="relu", input_shape = [2,]),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(1, activation = "sigmoid")
])

model_8.compile(loss = "BinaryCrossentropy",
                optimizer = "Adam",
                metrics = ["accuracy"])

# create a learning rate callback
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(lambda epoch: 1e-4 * 10**(epoch/20))
history_model_8 = model_8.fit(x_train, y_train, epochs = 50,
                              callbacks = [lr_scheduler], verbose=0)


In [ ]:
pd.DataFrame(history_model_8.history).plot(figsize=(10,7), xlabel = "epochs");

In [ ]:
# plot learning rate vs loss

lrs = 1e-4 * (10**(tf.range(50)/20))
plt.figure(figsize=(10,7))
plt.semilogx(lrs, history_model_8.history["loss"])
plt.xlabel("Learning Rate")
plt.ylabel("Loss")
plt.title("Learning rate vs. Loss");

# our ideal learning rate is where loss is reducing the fastest

In [ ]:
model_9 = tf.keras.Sequential([
    tf.keras.layers.Dense(4, activation="relu", input_shape = [2,]),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(1, activation = "sigmoid")
])

model_9.compile(loss = "BinaryCrossentropy",
                optimizer = tf.keras.optimizers.Adam(learning_rate = 0.02),
                metrics = ["accuracy"])

history_model_9 = model_9.fit(x_train, y_train, epochs = 25, verbose = 0)

In [ ]:
model_7.evaluate(x_test, y_test)

In [ ]:
model_9.evaluate(x_test, y_test)

# loss, acc is improved in the same amount of epochs